# Transformer Concepts: Annotated Reference
# Pure Python — No PyTorch, No NumPy
#
# This notebook is a "cheat-sheet" of every concept from Phase 6,
# with each operation spelled out as a loop so the math is impossible to hide.
# Use it as a reference when the PyTorch code in the main notebooks
# feels too abstract.

import math
import random

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PART A: THE MATH PRIMITIVES
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("PART A: Math Primitives\n")

# ─── Dot product ────────────────────────────────────────────
# a · b = a[0]*b[0] + a[1]*b[1] + ...
# This is the "similarity" measure: large when vectors point in same direction

def dot(a, b):
    return sum(x*y for x, y in zip(a, b))

a = [1.0, 0.0, 0.0]
b = [1.0, 0.0, 0.0]
c = [0.0, 1.0, 0.0]
print(f"dot(a, a) = {dot(a, a):.1f}  ← same vector: maximum similarity")
print(f"dot(a, c) = {dot(a, c):.1f}  ← orthogonal: no similarity")

# ─── Softmax ────────────────────────────────────────────────
# Converts any list of numbers into a probability distribution [0,1] summing to 1.
# The largest number gets the biggest share, others get smaller shares.

def softmax(scores):
    m    = max(s for s in scores if s != float('-inf'))
    exps = [math.exp(s - m) if s != float('-inf') else 0.0 for s in scores]
    t    = sum(exps)
    return [e/t if t > 0 else 0.0 for e in exps]

raw_scores = [2.0, 1.0, 0.5, -1.0]
probs      = softmax(raw_scores)
print(f"\nsoftmax({raw_scores})")
print(f"      = {[round(p, 4) for p in probs]}")
print(f"  sum = {sum(probs):.6f}  ← always exactly 1.0")

# ─── Matrix multiplication ──────────────────────────────────
# A (m×k) @ B (k×n) → C (m×n)
# C[i][j] = dot(row i of A, column j of B)

def mat_mul(A, B):
    m, k, n = len(A), len(A[0]), len(B[0])
    return [
        [sum(A[i][p] * B[p][j] for p in range(k)) for j in range(n)]
        for i in range(m)
    ]

A = [[1,2],[3,4]]
B = [[5,6],[7,8]]
print(f"\n[[1,2],[3,4]] @ [[5,6],[7,8]] = {mat_mul(A, B)}")
print(f"  expected:                    = [[19,22],[43,50]]")

# ─── Cross-Entropy Loss ─────────────────────────────────────
# Measures how surprised the model is by the correct answer.
# Low loss: model was confident and RIGHT.
# High loss: model was confident and WRONG (or unconfident).

def cross_entropy(logits, correct_index):
    probs = softmax(logits)
    return -math.log(max(probs[correct_index], 1e-9))

logits  = [2.0, 0.5, -1.0]  # 3 classes
correct = 0                   # Index 0 is the right answer
loss    = cross_entropy(logits, correct)
probs   = softmax(logits)
print(f"\ncross_entropy([2.0, 0.5, -1.0], correct_class=0)")
print(f"  softmax probs = {[round(p, 4) for p in probs]}")
print(f"  p(correct)    = {probs[correct]:.4f}")
print(f"  loss          = -log({probs[correct]:.4f}) = {loss:.4f}")
print(f"  (lower is better — perfect prediction → loss = 0)")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PART B: ATTENTION — STEP BY STEP
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n\nPART B: Attention Step by Step\n")

# The full attention formula:
#   Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
#
# Broken down into plain English:
#
#   Q·Kᵀ   : "How similar is my question to each piece of information?"
#   /√d_k  : "Scale down so softmax doesn't collapse to one-hot."
#   softmax: "Turn similarities into weights (probabilities) summing to 1."
#   ·V     : "Blend the actual information by those weights."

def attention_step_by_step(Q, K, V, mask=None, verbose=True):
    seq_q, d_k = len(Q), len(Q[0])
    d_v        = len(V[0])

    if verbose:
        print(f"  Inputs: Q={seq_q} queries, K={len(K)} keys, V={len(V)} values")
        print(f"  d_k={d_k}, d_v={d_v}\n")

    # STEP 1: Compute similarity scores Q @ K^T
    # scores[i][j] = "how much should query i attend to key j?"
    K_T    = [[K[j][d] for j in range(len(K))] for d in range(d_k)]  # transpose
    scores = mat_mul(Q, K_T)  # [seq_q, seq_k]
    if verbose:
        print(f"  Step 1 – Raw scores Q·Kᵀ:")
        for i, row in enumerate(scores):
            print(f"    Query {i}: {[round(s, 3) for s in row]}")

    # STEP 2: Scale
    scale  = math.sqrt(d_k)
    scores = [[s / scale for s in row] for row in scores]
    if verbose:
        print(f"\n  Step 2 – Scaled (/√{d_k}={scale:.2f}):")
        for i, row in enumerate(scores):
            print(f"    Query {i}: {[round(s, 3) for s in row]}")

    # STEP 3: Mask (optional — set blocked positions to -∞)
    if mask is not None:
        for i in range(seq_q):
            for j in range(len(K)):
                if mask[i][j] == 0:
                    scores[i][j] = float('-inf')
        if verbose:
            print(f"\n  Step 3 – After masking (blocked=−∞):")
            for i, row in enumerate(scores):
                display = [round(s, 3) if s != float('-inf') else '-inf' for s in row]
                print(f"    Query {i}: {display}")

    # STEP 4: Softmax → attention weights
    weights = [softmax(row) for row in scores]
    if verbose:
        print(f"\n  Step 4 – Attention weights (each row sums to 1):")
        for i, row in enumerate(weights):
            print(f"    Query {i}: {[round(w, 4) for w in row]}  sum={sum(row):.6f}")

    # STEP 5: Weighted sum of Values
    output = [[sum(weights[i][j] * V[j][d] for j in range(len(V))) for d in range(d_v)]
              for i in range(seq_q)]
    if verbose:
        print(f"\n  Step 5 – Output (weighted blend of V):")
        for i, row in enumerate(output):
            print(f"    Token {i}: {[round(v, 4) for v in row]}")

    return output, weights


# Run it on a 3-token example
random.seed(0)
seq, d_k, d_v = 3, 4, 4
Q = [[random.uniform(-1,1) for _ in range(d_k)] for _ in range(seq)]
K = [[random.uniform(-1,1) for _ in range(d_k)] for _ in range(seq)]
V = [[random.uniform(-1,1) for _ in range(d_v)] for _ in range(seq)]

print("── Full attention (no mask) ─────────────────────────────")
output, weights = attention_step_by_step(Q, K, V)

# Now with causal mask
print("\n── Causal attention (lower-triangular mask) ─────────────")
causal_mask = [[1 if j <= i else 0 for j in range(seq)] for i in range(seq)]
output_c, weights_c = attention_step_by_step(Q, K, V, mask=causal_mask)

# Verify: token 0 must attend ONLY to itself
assert weights_c[0][1] < 1e-9 and weights_c[0][2] < 1e-9, "Causal mask failed!"
print("\n  ✓ Token 0 correctly attends only to itself")
print(f"    weights_c[0] = {[round(w, 4) for w in weights_c[0]]}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PART C: KEY TRANSFORMER CONCEPTS AS RUNNABLE ASSERTIONS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n\nPART C: Key Concepts Verified with Assertions\n")

# ─── Concept 1: Attention weights always sum to 1 per query ─
print("Concept 1: Attention weight rows sum to 1")
for i, row in enumerate(weights):
    s = sum(row)
    assert abs(s - 1.0) < 1e-9, f"Row {i} sums to {s}, not 1!"
    print(f"  Row {i} sum: {s:.10f} ✓")

# ─── Concept 2: Causal mask zeros out the future ─────────────
print("\nConcept 2: Causal mask blocks future tokens")
for i in range(seq):
    for j in range(seq):
        if j > i:
            assert weights_c[i][j] < 1e-9, f"Future token leakage at [{i}][{j}]!"
print("  All future positions are 0.0 ✓")

# ─── Concept 3: Effect of temperature on generation ──────────
print("\nConcept 3: Temperature controls distribution sharpness")
logits_gen = [3.0, 2.5, 2.0, 1.5, 1.0]

for temp in [0.1, 0.5, 1.0, 2.0]:
    scaled = [l / temp for l in logits_gen]
    probs  = softmax(scaled)
    max_p  = max(probs)
    entropy = -sum(p * math.log(p + 1e-9) for p in probs)
    print(f"  temp={temp:>4}: max_prob={max_p:.4f}, entropy={entropy:.4f}  "
          f"{'(peaked/deterministic)' if max_p > 0.8 else '(spread/random)' if max_p < 0.4 else '(balanced)'}")

# ─── Concept 4: LayerNorm normalizes each token ──────────────
print("\nConcept 4: LayerNorm makes each token have mean≈0, std≈1")
token = [10.0, 500.0, 3.0, 250.0]  # Wildly different scales

mean = sum(token) / len(token)
var  = sum((v - mean)**2 for v in token) / len(token)
std  = math.sqrt(var + 1e-6)
normed = [(v - mean) / std for v in token]

print(f"  Before: {[round(v, 1) for v in token]}")
print(f"  After:  {[round(v, 4) for v in normed]}")
print(f"  mean = {sum(normed)/len(normed):.8f}  (≈ 0)")
print(f"  std  = {math.sqrt(sum((v - sum(normed)/len(normed))**2 for v in normed) / len(normed)):.8f}  (≈ 1)")

# ─── Concept 5: Residual connection preserves information ────
print("\nConcept 5: Residual connection — block learns the CHANGE, not the full value")

original   = [1.0, 2.0, 3.0]
change     = [0.1, -0.05, 0.2]   # Small delta from the attention/FFN sublayer
output_res = [o + c for o, c in zip(original, change)]  # x + SubLayer(x)

print(f"  Original (x):       {original}")
print(f"  SubLayer output:    {change}")
print(f"  After residual add: {output_res}")
print(f"  The block only needed to learn a small correction (0.1, -0.05, 0.2)")
print(f"  not reconstruct the full output from scratch.")
print(f"  This is WHY skip connections make training stable — the gradient has")
print(f"  a direct path through the '+' operation back to earlier layers.")

# ─── Concept 6: Weight tying — embedding ≈ inverse of lm_head ─
print("\nConcept 6: Weight tying intuition")
print("  The token embedding maps integers → vectors:  'dog' → [0.2, -0.5, 0.8, ...]")
print("  The lm_head maps vectors → integers:  [0.2, -0.5, 0.8, ...] → 'dog' (predicted)")
print("  These are inverses of each other → sharing their weights saves parameters")
print("  AND improves performance (tokens with similar embeddings get similar predictions).")
print()
d = 4
vocab = 5
# Embedding table: each row = a token's representation
emb_table = [[random.gauss(0, 0.1) for _ in range(d)] for _ in range(vocab)]

# Instead of a separate lm_head weight matrix, REUSE the embedding table (transposed)
# This is weight tying: lm_head.W = embedding_table.T
lm_head_W = [[emb_table[v][d_] for v in range(vocab)] for d_ in range(d)]

# hidden[t] (a d-dimensional vector) → vocab scores
hidden_vec = [0.3, -0.1, 0.2, 0.5]
vocab_scores = [dot(hidden_vec, emb_table[v]) for v in range(vocab)]

print(f"  Hidden vector: {[round(v, 2) for v in hidden_vec]}")
print(f"  Vocab scores (via tied embedding): {[round(s, 4) for s in vocab_scores]}")
print(f"  → Token {vocab_scores.index(max(vocab_scores))} has highest score (predicted next token)")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PART D: WHAT HAPPENS INSIDE DURING TEXT GENERATION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n\nPART D: Text Generation Loop Traced by Hand\n")

# Simulate generating 3 tokens from a 2-token prompt
# using a tiny toy vocabulary [a, b, c, d] = [0, 1, 2, 3]

vocab    = ['a', 'b', 'c', 'd']
prompt   = [0, 1]  # "ab"
context  = list(prompt)

print(f"Vocab: {dict(enumerate(vocab))}")
print(f"Prompt: {[vocab[i] for i in prompt]} (ids={prompt})\n")

random.seed(5)

for step in range(3):
    print(f"  Generation step {step+1}:")
    print(f"    Context so far: {''.join(vocab[i] for i in context)} (ids={context})")

    # (In a real model: forward(context) → logits)
    # Here we use random logits to simulate a model output
    fake_logits = [random.uniform(-1, 2) for _ in vocab]
    print(f"    Logits for next token: {[round(l, 3) for l in fake_logits]}")

    # Convert to probabilities
    probs = softmax(fake_logits)
    print(f"    Probabilities:         {[round(p, 4) for p in probs]}")

    # Weighted random sample (temperature=1.0)
    r, cumulative = random.random(), 0.0
    next_id = len(vocab) - 1
    for tok_id, p in enumerate(probs):
        cumulative += p
        if r < cumulative:
            next_id = tok_id
            break

    print(f"    Sampled:  '{vocab[next_id]}' (id={next_id}, p={probs[next_id]:.4f})")
    context.append(next_id)
    print(f"    Context: {''.join(vocab[i] for i in context)}\n")

print(f"Final generated sequence: {''.join(vocab[i] for i in context)}")
print(f"  Prompt:    {''.join(vocab[i] for i in prompt)}")
print(f"  Generated: {''.join(vocab[i] for i in context[len(prompt):])}")